# ⚙️ Football Feature Engineering Pipeline

This notebook focuses on the creation and transformation of analytical features used to enhance football match prediction models and performance analysis.

The feature engineering process aims to extract meaningful competitive indicators from historical football data while improving predictive power and dataset interpretability.

---

## 📌 Main Objectives

- Create advanced team performance features
- Build weighted historical metrics
- Develop offensive and defensive indicators
- Generate temporal and contextual variables
- Prepare structured inputs for machine learning models

---


In [ ]:
# 🎨 Visualization Style Configuration

import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

sns.set_theme(style="whitegrid", palette="viridis")


# ============================================
# 🌍 FIFA World Cup Project
# ⚙️ Feature Engineering
# ============================================

---

In [1]:
# ============================================
# 📚 Import Libraries
# ============================================

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

In [2]:
# ============================================
# 📥 Load Clean Datasets
# ============================================

results = pd.read_csv(
    "../data/processed/results_clean.csv"
)

wcmatches = pd.read_csv(
    "../data/processed/wcmatches_clean.csv"
)

worldcups = pd.read_csv(
    "../data/processed/worldcups_clean.csv"
)

In [ ]:
# 🇩🇪 Data Standardization: Germany Unification

# Automatically detect dataframe variables
possible_dfs = ["df", "matches", "results", "world_cup", "team_metrics", "data"]

for df_name in possible_dfs:

    if df_name in globals():

        dataframe = globals()[df_name]

        country_columns = [
            col for col in dataframe.columns
            if any(keyword in col.lower()
                   for keyword in ["team", "country", "winner", "home", "away"])
        ]

        for col in country_columns:
            dataframe[col] = dataframe[col].replace("West Germany", "Germany")

        print(f"✅ Germany standardization applied to: {df_name}")


## 📅 Temporal Features

Extracting time-based variables and historical progression patterns.

In [3]:
# ============================================
# 📅 Date Formatting
# ============================================

results["date"] = pd.to_datetime(
    results["date"]
)

results["year"] = (
    results["date"]
    .dt.year
)

In [4]:
# ============================================
# 🏆 Match Result Variable
# ============================================

def get_result(row):

    if row["home_score"] > row["away_score"]:
        return "Home Win"

    elif row["home_score"] < row["away_score"]:
        return "Away Win"

    else:
        return "Draw"

results["match_result"] = (
    results.apply(get_result, axis=1)
)

results["match_result"].value_counts()

match_result
Home Win    23754
Away Win    13665
Draw        11170
Name: count, dtype: int64

## 🗂️ Dataset Overview

Loading and validating datasets used for feature engineering.

In [5]:
# ============================================
# ⚽ Goal Difference
# ============================================

results["goal_difference"] = (
    results["home_score"] -
    results["away_score"]
)

results[
    [
        "home_score",
        "away_score",
        "goal_difference"
    ]
].head()

,home_score,away_score,goal_difference
0,0.0,0.0,0.0
1,4.0,2.0,2.0
2,2.0,1.0,1.0
3,2.0,2.0,0.0
4,3.0,0.0,3.0


## 📈 Team Performance Indicators

Analyzing offensive, defensive, and consistency-related features.

In [6]:
# ============================================
# 🔥 Total Goals
# ============================================

results["total_goals"] = (
    results["home_score"] +
    results["away_score"]
)

results[
    [
        "home_score",
        "away_score",
        "total_goals"
    ]
].head()

,home_score,away_score,total_goals
0,0.0,0.0,0.0
1,4.0,2.0,6.0
2,2.0,1.0,3.0
3,2.0,2.0,4.0
4,3.0,0.0,3.0


In [7]:
# ============================================
# 🏠 Home Advantage
# ============================================

results["home_advantage"] = (
    results["neutral"]
    .apply(lambda x: 0 if x else 1)
)

results[
    [
        "neutral",
        "home_advantage"
    ]
].head()

,neutral,home_advantage
0,False,1
1,False,1
2,False,1
3,False,1
4,False,1


## 🏋️ Weighted Performance Metrics

Building historical weighted metrics to capture recent team form.

In [8]:
# ============================================
# 🏆 Tournament Importance
# ============================================

tournament_weights = {

    "Friendly": 1,

    "FIFA World Cup qualification": 3,
    "UEFA Euro qualification": 3,
    "AFC Asian Cup qualification": 3,

    "UEFA Nations League": 4,

    "Copa América": 5,
    "UEFA Euro": 5,

    "FIFA World Cup": 10
}

results["tournament_weight"] = (
    results["tournament"]
    .map(tournament_weights)
    .fillna(2)
)

results[
    [
        "tournament",
        "tournament_weight"
    ]
].head()

,tournament,tournament_weight
0,Friendly,1.0
1,Friendly,1.0
2,Friendly,1.0
3,Friendly,1.0
4,Friendly,1.0


In [10]:
# ============================================
# ⏳ Temporal Weight
# ============================================

current_year = results["year"].max()

results["time_weight"] = (
    1 + (
        (results["year"] - current_year)
        / 100
    )
)

results[
    [
        "year",
        "time_weight"
    ]
].head()

,year,time_weight
0,1872,-0.54
1,1873,-0.53
2,1874,-0.52
3,1875,-0.51
4,1876,-0.50


In [11]:
# ============================================
# 🔥 Combined Match Weight
# ============================================

results["combined_weight"] = (
    results["tournament_weight"] *
    results["time_weight"]
)

results[
    [
        "tournament_weight",
        "time_weight",
        "combined_weight"
    ]
].head()

,tournament_weight,time_weight,combined_weight
0,1.0,-0.54,-0.54
1,1.0,-0.53,-0.53
2,1.0,-0.52,-0.52
3,1.0,-0.51,-0.51
4,1.0,-0.50,-0.50


## ⚙️ Feature Engineering

Creating advanced analytical variables and performance indicators.

In [12]:
# ============================================
# ✅ Feature Validation
# ============================================

results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,match_result,goal_difference,total_goals,home_advantage,tournament_weight,time_weight,combined_weight
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1872,Draw,0.0,0.0,1,1.0,-0.54,-0.54
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1873,Home Win,2.0,6.0,1,1.0,-0.53,-0.53
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1874,Home Win,1.0,3.0,1,1.0,-0.52,-0.52
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1875,Draw,0.0,4.0,1,1.0,-0.51,-0.51
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1876,Home Win,3.0,3.0,1,1.0,-0.50,-0.50


In [13]:
# ============================================
# 💾 Save Feature Dataset
# ============================================

results.to_csv(
    "../data/processed/results_features.csv",
    index=False
)

print("✅ Feature dataset saved successfully.")

✅ Feature dataset saved successfully.


💡 📌 Observation:

The feature engineering process transformed raw football match data into structured analytical variables suitable for predictive modeling.

Features such as goal difference, home advantage, tournament importance, and temporal weighting capture both contextual and performance-related aspects of international football matches, significantly improving the dataset’s analytical value for future machine learning tasks.

---

🏁 🎯 Conclusion:

The engineered dataset now contains meaningful football intelligence features that better represent team strength, match importance, and competitive context.

These features establish the foundation for predictive modeling, FIFA ranking simulations, and World Cup outcome forecasting in the next stages of the project.

---